<a href="https://colab.research.google.com/github/cvandekerckh/mlsmm2151/blob/main/chapitres/ch01-projet-fil-rouge/01-paradigmes-donnees-et-questions.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Chapitre 1 : Le projet fil rouge

*MLSMM2151 Data Science · Séance 1, vendredi 13 novembre, matin · Corentin Vande Kerckhove*

> **CONSEIL** : commencez par enregistrer une copie de ce notebook dans votre Google Drive
> (`Fichier` > `Enregistrer une copie dans Drive`) pour pouvoir prendre vos propres notes.

## Notebook 1 sur 1 : la data science, les paradigmes et le projet fil rouge

**Durée estimée : 45 minutes.**

### Objectifs

À la fin de ce notebook, vous saurez :

- expliquer ce qu'un algorithme d'apprentissage fait de différent d'une règle
  écrite à la main, et pourquoi cela a changé la façon de décider ;
- dire, face à une demande métier, de quel paradigme elle relève, et s'il
  s'agit d'une régression ou d'une classification ;
- énoncer les deux questions métier qui structurent tout le cours ;
- charger un jeu de données réel et en tirer trois observations en cinq minutes ;
- lire la distribution d'une cible et en tirer le premier chiffre à retenir.

## 1.1. La data science aujourd'hui, et ce qu'un algorithme y fait

Vous en avez utilisé ce matin sans y penser. Le prix du billet de train que
vous avez payé, l'ordre des résultats d'une recherche, la vidéo proposée
ensuite, le message classé en spam, le virement bloqué pour vérification, le
nombre de caisses ouvertes au supermarché : derrière chacune de ces décisions,
quelqu'un a confié à un programme le soin de trancher, des milliers de fois par
jour, à partir de données.

C'est cela, la data science en pratique : **transformer des données en
décisions**, et le faire de façon répétable. Pas un tableau de bord qu'on
regarde, une décision qu'on prend, ou qu'on laisse prendre.

### Deux façons d'écrire une règle

Prenons une question concrète, celle qui nous occupera tout le cours : *à quel
prix mettre en ligne un logement ?* Un gestionnaire expérimenté a sa règle :
« trente euros par personne, à peu près ». C'est une **règle écrite à la
main** : quelqu'un l'a formulée, à partir de son expérience.

Un **algorithme d'apprentissage** fait autre chose. On ne lui donne pas la
règle ; on lui donne des **exemples**, des logements dont on connaît la
capacité et le prix, et une **procédure** pour en tirer la règle qui colle le
mieux à ces exemples. La règle n'est plus écrite par un humain : elle est
*apprise*, et elle change si les exemples changent.

Voyons les deux à l'œuvre sur huit logements inventés pour l'exemple, puis sur
un neuvième, de cinq personnes, dont il faut fixer le prix.

In [ ]:
import pandas as pd
from sklearn.linear_model import LinearRegression

# Huit logements inventés pour l'exemple : capacité d'accueil et prix par nuit.
exemples = pd.DataFrame({
    "accommodates": [1, 2, 2, 3, 4, 4, 6, 8],
    "prix":         [60, 75, 90, 100, 120, 130, 170, 210],
})
nouveau = pd.DataFrame({"accommodates": [5]})

# Règle écrite à la main : trente euros par personne.
prix_regle = 30 * nouveau["accommodates"][0]

# Règle apprise : l'algorithme ajuste la droite qui colle le mieux aux huit exemples.
modele = LinearRegression().fit(exemples[["accommodates"]], exemples["prix"])
prix_appris = modele.predict(nouveau)[0]

print(f"Règle à la main : {prix_regle:.0f} euros")
print(f"Règle apprise   : {prix_appris:.0f} euros, "
      f"soit {modele.intercept_:.0f} euros fixes + {modele.coef_[0]:.0f} euros par personne")

In [ ]:
import matplotlib.pyplot as plt

# Les huit exemples et les deux règles sur une même figure. L'axe s'étend
# jusqu'au logement à tarifer, pour voir ce que chaque règle fait au-delà des exemples.
capacites = pd.DataFrame({"accommodates": range(0, max(10, nouveau["accommodates"][0]) + 1)})
fig, ax = plt.subplots(figsize=(7, 4))
ax.scatter(exemples["accommodates"], exemples["prix"], color="black", zorder=3, label="les huit exemples")
ax.plot(capacites["accommodates"], 30 * capacites["accommodates"], linestyle="--",
        label="règle à la main : 30 euros par personne")
ax.plot(capacites["accommodates"], modele.predict(capacites), label="règle apprise sur les exemples")
ax.scatter([nouveau["accommodates"][0]] * 2, [prix_regle, prix_appris], marker="x", s=80,
           color="red", zorder=4, label="le logement à tarifer, selon chaque règle")
ax.set_xlabel("capacité d'accueil (personnes)")
ax.set_ylabel("prix par nuit (euros)")
ax.legend()
plt.show()

Pour cinq personnes, les deux règles sont d'accord à quelques euros près : 150
contre 146. Mais regardez la règle apprise : elle n'a pas trouvé « tant par
personne », elle a trouvé **un coût fixe d'environ 39 euros**, plus 22 euros
par personne. Personne ne le lui a dit. Les huit exemples le contenaient, et la
procédure l'en a extrait.

La figure le montre : les deux droites se croisent vers cinq personnes, et
s'écartent de part et d'autre. En dessous, la règle à la main est la moins
chère ; au-dessus, c'est elle la plus chère. Hors de la zone des exemples, les
deux continuent tout droit, avec le même aplomb.

Trois choses à retenir de cet exemple, qui structurent tout le cours.

- **L'algorithme ne remplace pas la question, il y répond.** C'est nous qui
  avons décidé que le prix dépendait de la capacité, et que « coller aux
  exemples » voulait dire une droite. Ces choix sont les nôtres, et ce cours
  apprend à les faire et à les défendre.
- **La règle apprise vaut ce que valent les exemples.** Huit logements
  inventés, une seule variable : cette règle ne mérite aucune confiance. Avec
  6 721 vraies annonces et une trentaine de variables, la conversation change.
  C'est le projet fil rouge, en 1.3.
- **Une règle apprise se trompe autrement qu'une règle à la main.** Elle se
  trompe là où les exemples manquent, en silence, avec le même aplomb que là
  où ils abondent. Savoir où elle se trompe est la moitié du métier : c'est
  l'objet des chapitres 5, 7 et 8.

### Hack-Time 🛠️

Vous venez de comparer une règle à la main et une règle apprise sur un logement
de cinq personnes. À votre tour, dans la cellule de calcul plus haut (relancez-la, puis relancez
la figure) :

1. Remplacez `5` par `1` dans `nouveau`, relancez. Les deux règles ne sont plus
   d'accord du tout. Laquelle croyez-vous pour un studio, et pourquoi ?
2. Remplacez par `20`, relancez. Que vaut chacune des deux prédictions ? Y
   a-t-il un seul exemple qui ressemble à ce logement ?
3. En une phrase : qu'est-ce qui manque à ces huit exemples pour qu'on puisse
   faire confiance à la règle apprise ?

**Quatre minutes.**

**Défi optionnel**
Ajoutez un neuvième exemple aberrant, par exemple un logement de 2 personnes à
400 euros, et relancez pour 5 personnes. De combien la règle apprise a-t-elle
bougé ? La règle à la main, elle, n'a pas bougé : est-ce une qualité ou un
défaut ?

*Vos réponses :*

1.
2.
3.

## 1.2. Les trois paradigmes, en pratique

Data Analytics vous a présenté les trois familles. Un rappel en trois lignes,
puis l'exercice : la difficulté n'est pas de réciter les définitions, c'est de
reconnaître le paradigme derrière une demande formulée en langage métier.

- **Supervisé** : on dispose d'exemples où **la réponse est connue**. Chapitres
  5 à 10. Deux sous-familles, selon la nature de la réponse :

| | Ce qu'on prédit | Exemple chez nous | Exemples ailleurs |
|---|---|---|---|
| **Régression** | Un nombre | Le prix d'une annonce | Chiffre d'affaires, délai de livraison, consommation |
| **Classification** | Une catégorie | L'annonce est-elle Superhost | Client qui part ou non, mail spam ou non, défaut de paiement |

- **Non supervisé** : **aucune réponse** connue, on cherche une structure.
  Personne ne peut dire si le résultat est *juste*, seulement s'il est
  *utile* : c'est ce qui le rend difficile à évaluer. Chapitres 11 et 12.
- **Par renforcement** : un agent apprend en agissant et en recevant des
  récompenses. Il faut pouvoir expérimenter en vrai, ce qui est rare en
  gestion. Nous ne le traiterons pas.

![Les trois paradigmes](https://raw.githubusercontent.com/cvandekerckh/mlsmm2151/main/images/ch01-paradigmes.png)

*Les trois familles, et la question qui les départage.*

La distinction régression ou classification n'est pas cosmétique : elle change
les modèles utilisables et, surtout, la façon de mesurer si le modèle est bon.
C'est l'objet du chapitre 7.

### Hack-Time 🛠️

Six demandes. Pour chacune : **supervisé (régression), supervisé
(classification), non supervisé, ou renforcement** ?

1. Estimer la durée de traitement d'une commande à partir de son contenu.
2. Regrouper 40 000 clients en profils d'achat comparables, sans idée préalable
   des profils.
3. Détecter si un paiement est frauduleux, à partir de 200 000 paiements dont on
   sait lesquels l'étaient.
4. Ajuster le prix d'un billet d'avion heure par heure pour maximiser le revenu
   du vol.
5. Repérer les 50 transactions les plus atypiques du mois, sans liste de fraudes
   connues.
6. Prévoir le montant qu'un client dépensera l'année prochaine.

**Six minutes.** Attention, deux d'entre elles se ressemblent beaucoup et ne
relèvent pas du même paradigme.

**Défi optionnel**
Comparez les cas 3 et 5. Ils portent tous les deux sur des transactions
suspectes. Qu'est-ce qui, concrètement, fait basculer l'un dans le supervisé et
l'autre dans le non supervisé ? Et si demain votre entreprise commence à
étiqueter ses fraudes, dans quel sens le problème bascule-t-il ?

*Vos réponses :*

| # | Paradigme | Pourquoi |
|---|---|---|
| 1 | | |
| 2 | | |
| 3 | | |
| 4 | | |
| 5 | | |
| 6 | | |

## 1.3. Le projet fil rouge : des données réelles, deux questions

![La Grand-Place de Bruxelles](https://raw.githubusercontent.com/cvandekerckh/mlsmm2151/main/images/ch01-bruxelles.jpg)

*Bruxelles, Grand-Place. Photo : Celuici, Wikimedia Commons, CC BY-SA 4.0.*

**Le contexte.** Vous êtes data scientist dans une société de gestion de
logements en location courte durée à Bruxelles. Votre interlocutrice est la
responsable du revenue management. Elle vous pose deux questions, et ce sont
elles que nous allons traiter jusqu'à la fin du cours.

> **Question 1.** « Quand un propriétaire nous confie un nouveau logement, à quel
> prix par nuit faut-il le mettre en ligne ? »

Un prix, donc un nombre : c'est un problème de **régression**. La cible s'appelle
`y_price` dans nos données.

> **Question 2.** « Parmi les logements que nous gérons, lesquels ont le profil
> pour décrocher le statut Superhost ? On voudrait concentrer nos efforts
> d'accompagnement sur ceux-là. »

Superhost ou pas : c'est un problème de **classification**. La cible s'appelle
`y_superhost`.

Un même jeu de données, deux questions de nature différente. C'est volontaire :
vous verrez que presque tout change entre les deux, y compris des choses qui ne
sont pas évidentes, comme la façon de mesurer la performance.

![Le projet fil rouge](https://raw.githubusercontent.com/cvandekerckh/mlsmm2151/main/images/ch01-fil-rouge.png)

*Le parcours des données sur le cours : des annonces brutes aux deux questions.*


### Les données, telles qu'elles arrivent

Tout le cours utilise le même jeu de données, pour que vous n'ayez à comprendre
qu'un seul contexte métier et que vous puissiez comparer ce que chaque méthode
apporte : les **6 721 annonces Airbnb** de Bruxelles, telles qu'elles étaient
le 21 juin 2025. Commençons par les données brutes, avant tout nettoyage.

Remarquez la ligne ci-dessous : on lit un fichier depuis une **adresse web**,
pas depuis un fichier local. Aucune donnée n'est stockée dans le dépôt du cours.
Le chapitre 2 explique pourquoi, en détail.

In [ ]:
import pandas as pd

BASE = "https://github.com/cvandekerckh/mlsmm2151/releases/download/v1-donnees"

annonces = pd.read_csv(f"{BASE}/airbnb_bruxelles_raw.csv", low_memory=False)

print(f"{annonces.shape[0]} annonces, {annonces.shape[1]} colonnes")
annonces.head(3)

Regardons ce que pandas pense de ces colonnes.

In [ ]:
annonces.info()

Prenez trente secondes pour lire cette sortie. Deux choses devraient vous
surprendre.

D'abord, `price` est de type `object` (ou `str`, selon la version de pandas),
c'est-à-dire du texte. Regardons.

In [ ]:
annonces[["price", "host_response_rate", "host_since", "host_is_superhost"]].head()

Voilà le vrai visage des données.

- `price` vaut `$64.00` : un prix stocké en texte, avec un symbole monétaire.
  Impossible d'en faire une moyenne en l'état.
- `host_response_rate` vaut `100%` : un pourcentage en texte.
- `host_since` vaut `2010-08-11` : une date en texte, que pandas ne sait pas
  ordonner correctement.
- `host_is_superhost` vaut `t` : un booléen déguisé.

Ce n'est pas un défaut de ce jeu de données en particulier. **C'est l'état
normal des données réelles.** Le chapitre 3 est entièrement consacré à cela.

### Hack-Time 🛠️

Vous venez de voir que `annonces.info()` révèle des problèmes de type, et que
`annonces.head()` révèle des problèmes de contenu.

À votre tour, dans la cellule ci-dessous :

1. Affichez le taux de valeurs manquantes de chaque colonne, trié du plus
   manquant au moins manquant. Une piste : `annonces.isna().mean()` donne la
   proportion, et `.sort_values(ascending=False)` la trie.
2. Repérez **la colonne qui ne contient aucune information du tout**.
3. En une phrase : que fait-on d'une colonne entièrement vide, et pourquoi
   est-elle facile à rater ?

**Six minutes.**

**Défi optionnel**
Les colonnes `review_scores_rating`, `reviews_per_month` et `first_review` ont
exactement le même taux de manquants. Ce n'est pas un hasard. Quelle est
l'explication, et qu'est-ce que cela change pour la suite ?

In [ ]:
# Votre code ici

### Les deux cibles, dans la table de modélisation

Les chapitres 5 à 12 travaillent sur une version déjà nettoyée de ces annonces,
découpée en un jeu d'entraînement et un jeu de test, pour que la modélisation
ne dépende pas de la qualité du nettoyage de chacun. Regardons-y nos deux
cibles.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

BASE = "https://github.com/cvandekerckh/mlsmm2151/releases/download/v1-donnees"

train = pd.read_csv(f"{BASE}/airbnb_bruxelles_train.csv")
# La table de modélisation, déjà nettoyée : elle sert aux chapitres 5 à 12.
print(f"{train.shape[0]} annonces, {train.shape[1]} colonnes")

In [ ]:
# Question 1 : le prix. Une variable continue.
print(train["y_price"].describe().round(1).to_string())

In [ ]:
# Question 2 : le statut Superhost. Une variable binaire.
repartition = train["y_superhost"].value_counts(normalize=True).sort_index()
print(repartition.rename({0: "Pas Superhost", 1: "Superhost"}).round(3).to_string())

Deux chiffres à retenir dès maintenant, parce qu'ils reviendront.

**Le prix médian est de 88 euros, la moyenne de 107.** La moyenne est nettement
au-dessus de la médiane : la distribution est étirée vers la droite par quelques
logements chers. Regardons.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))

axes[0].hist(train["y_price"], bins=60, color="#4C72B0", edgecolor="white")
axes[0].axvline(train["y_price"].median(), color="#C44E52", linestyle="--", label="médiane")
axes[0].axvline(train["y_price"].mean(), color="#DD8452", linestyle="--", label="moyenne")
axes[0].set_title("Prix par nuit")
axes[0].set_xlabel("euros")
axes[0].legend()

axes[1].bar(["Pas Superhost", "Superhost"], repartition.values, color=["#CCCCCC", "#4C72B0"])
axes[1].set_title("Statut Superhost")
axes[1].set_ylabel("proportion")

plt.tight_layout()
plt.show()

**Seulement 27,6 % des annonces sont Superhost.** Les deux classes ne sont pas
équilibrées. Gardez ce chiffre en tête : au chapitre 7, il expliquera pourquoi un
modèle qui répond systématiquement « pas Superhost » obtient 72 % de bonnes
réponses tout en étant strictement inutile.

### Hack-Time 🛠️

On vient de décrire les deux cibles. À votre tour d'aller chercher un premier
élément de réponse métier, dans la cellule ci-dessous.

1. Calculez le **prix médian par type de logement** (`room_type`).
   Une piste : `train.groupby("room_type")["y_price"].median()`.
2. Ajoutez le **nombre d'annonces** de chaque type, pour savoir si vous pouvez
   faire confiance à ces médianes. `.agg(["median", "count"])` fait les deux
   d'un coup.
3. En une phrase : que répondriez-vous à la responsable du revenue management
   si elle vous demandait maintenant le prix d'un `Private room` ?

**Six minutes.**

**Défi optionnel**
Refaites le calcul par commune (`neighbourhood_cleansed`) et triez par prix
médian décroissant. La commune la plus chère vous surprend-elle ? Regardez son
nombre d'annonces avant de conclure quoi que ce soit.

In [ ]:
# Votre code ici

## À retenir

- Un algorithme d'apprentissage ne reçoit pas la règle, il la tire des exemples.
  La règle vaut ce que valent les exemples, et c'est nous qui posons la question.
- Dans le supervisé, prédire un **nombre** est une régression, prédire une
  **catégorie** est une classification. Cette distinction commande le choix du
  modèle et surtout la façon de l'évaluer.
- Notre fil rouge porte les deux : `y_price` en régression, `y_superhost` en
  classification, et ces deux questions nous occuperont jusqu'à la fin du cours.
- Les données réelles arrivent sales. Un prix qui vaut `$64.00` n'est pas un
  accident, c'est la norme. Le chapitre 3 y est entièrement consacré.
- Constater n'est pas décider : on explore avant de nettoyer, pour pouvoir
  justifier chaque choix.
- Les classes de `y_superhost` sont déséquilibrées, 27,6 % contre 72,4 %. Ce
  déséquilibre nous posera un vrai problème au chapitre 7.

**La suite** : le chapitre 2 traite de la reproductibilité. Comment organiser un
projet, où vivent les données, et comment utiliser Git et GitHub. C'est le
chapitre qui vous servira dans tous les devoirs.